# Assignment: Extend the az.ipynb Lab

**Based on:** `Lab2.ipynb` (the Module 3 lab).

This week's assignment is short on purpose: take your working `Lab2.ipynb` lab and add **one
more step** to the chain. No new concepts, no new setup, no new libraries — just one more
chained LLM call that builds on what you already have.

**Two deployments this time:** `gpt-5.1-ptu` is the default deployment for every existing
step (Steps 1–4). The new step you add (Step 5) must call `gpt-5.4-ptu` instead.

## Step 1 — Start from your working lab

- Make a copy of your completed `Lab2.ipynb` (e.g. rename the copy `assignment3.ipynb`), or
  continue directly inside this notebook — either is fine.
- Copy in your working code from the lab's Steps 1–4: the imports and `.env` config, the
  `AzureOpenAI` client, the `chat()` helper, and the chain itself (fun fact → generate a hard
  question → answer it → evaluate the answer).
- Confirm your `.env`'s `AZURE_APIM_OPENAI_DEPLOYMENT` is set to `gpt-5.1-ptu` — this stays
  the default deployment for Steps 1–4, unchanged.

Run those cells first and confirm they still work before moving on.

In [5]:
# TODO: paste your working Lab2.ipynb code here (Steps 1-4):
#   - imports + load_dotenv + config variables (AZURE_APIM_OPENAI_DEPLOYMENT = "gpt-5.1-ptu")
#   - the AzureOpenAI client
#   - the chat() helper
#       NOTE: give chat() an optional `deployment` parameter that defaults to the
#       module-level `deployment` variable, so a single call can override it later:
#           def chat(messages, max_completion_tokens=5000, deployment=deployment):
#               return client.chat.completions.create(
#                   model=deployment, messages=messages,
#                   max_completion_tokens=max_completion_tokens,
#               )
#   - the chain: fact -> question -> answer -> evaluation (all using the default deployment)
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI

# Read .env and override any existing process env values.
load_dotenv(override=True)

# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")

if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")

# Sync client pointed at Azure APIM.
openai = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)
def chat(messages, max_completion_tokens=1000, deployment=deployment):
    return openai.chat.completions.create(
        model=deployment,
        messages=messages,
        max_completion_tokens=max_completion_tokens,
    )

Azure APIM key exists and begins 5b21e970
Deployment: gpt-5.1-ptu


## Step 2 — Add one more chained step

Add a **5th step** to the chain. Its prompt must be built from at least one variable you
already have (`fact`, `question`, `answer`, or the evaluation text) — the same chaining
pattern as every other step in the lab.

**This step must call `gpt-5.4-ptu`, not the default deployment.** Pass it explicitly when
you call `chat()`:

```python
response = chat(messages, deployment="gpt-5.4-ptu")
```

Pick **one** idea below, or invent your own:

- Rate the difficulty of the question on a 1–10 scale, with a one-sentence justification.
- Rewrite the answer in one simple sentence a 10-year-old could understand.
- Suggest one new, related fun fact that connects to the original topic.
- Translate the final answer into a language of your choice.
- Write a one-line verdict on whether the model's own answer was actually correct, and why.

Store the result in its own variable, and print it clearly labeled (e.g. `=== STEP 5
(gpt-5.4-ptu) ===`).

In [6]:
# TODO: Step 5 - build a new prompt using an earlier variable
#   (fact, question, answer, and/or the evaluation)
# TODO: call chat(messages, deployment="gpt-5.4-ptu") -- do NOT use the default deployment here
# TODO: extract the result, and print it clearly labeled
messages = [{"role": "user", "content": "Tell me a fact that comes from Brasil"}]
response = chat(messages, deployment="gpt-5.4-ptu")
print("=== Step 5 (gpt-5.4-ptu) ===")
print(response.choices[0].message.content)

=== Step 5 (gpt-5.4-ptu) ===
Brazil is the world’s largest producer of coffee.


## Reflection

Answer in a sentence or two each:

1. **Which earlier variable(s) did your Step 5 prompt use, and why that one?**
2. **What would break if you ran Step 5 before the step it depends on?**
3. **Why might a real project deliberately use a different deployment (e.g. a stronger or
   more expensive model) for just one step in a chain, instead of using it everywhere?**

### My reflection

1. I used the Fact one because of its simple syntax and the amount of possible facts I could ask for him.

2. I would not work because is dependent on the variables that are created in the previous steps, so if you just jump straight to step 5 it would not have necessary information to generate that information.
3. Because using it would save time and cost since stronger deployments can take much more time to "think" and compile into a response, either giving you unecessary information or too much information, additionally some steps doesn't need high intelligent and strong deployments to perform their work optmizing the entire process by using cheaper and weaker ones that handle the job.

## Submission checklist

- [ ] Notebook runs top to bottom without errors (`Kernel → Restart & Run All`)
- [ ] `.env` file is **not** included in your submission
- [ ] Step 5 is clearly labeled and its prompt uses at least one earlier variable
- [ ] Reflection questions are answered